# Day 047 | DBSCAN & Hierarchical Clustering
## 100 Days of ML for Geotechnical Engineering
**Phase:** Classical Machine Learning

**Author:** Ripon Chandra Malo  
**Date:** 2026-05-16  

---

## What We Will Learn Today

| Concept | What It Means | Geotechnical Example |
|---------|--------------|---------------------|
| DBSCAN | Density-based clustering (no need to specify k) | Find soil zones based on natural data density |
| eps & min_samples | DBSCAN neighborhood parameters | How close and how many boreholes define a zone |
| Outlier detection | DBSCAN labels noise points as -1 | Identify anomalous boreholes that don't fit any zone |
| Hierarchical clustering | Build a tree of nested clusters | Show how soil zones merge from fine to coarse groupings |
| Dendrogram | Tree diagram of merge distances | Visual guide for choosing the number of soil zones |
| Algorithm comparison | K-Means vs DBSCAN vs Hierarchical | Which clustering fits spatial soil data best? |

---

## Our Geotechnical Problem

A site investigation involves **100 boreholes** spread across a construction site. Each borehole has:

- **x, y coordinates** (spatial location)
- **N-value** from SPT testing
- **Water content** (%)
- **Fines content** (%)

We need to identify **natural soil zones** across the site. Unlike K-Means, DBSCAN can find irregularly shaped zones and flag outlier boreholes. Hierarchical clustering shows how zones relate to each other at different scales.

---

## Part 1: Generate Spatial Soil Data

In [ ]:
# ─────────────────────────────────────────────────────
# IMPORTS
# ─────────────────────────────────────────────────────

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN, KMeans, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage

np.random.seed(42)
print("All imports successful.")

In [ ]:
# ─────────────────────────────────────────────────────
# GENERATE SPATIAL SOIL DATA WITH NATURAL CLUSTERS
# ─────────────────────────────────────────────────────

# Zone 1: Dense sand (NE quadrant) — tight cluster
n1 = 30
x1 = np.random.normal(80, 8, n1)
y1 = np.random.normal(75, 10, n1)
N1 = np.random.normal(35, 5, n1).clip(20, 50)
wc1 = np.random.normal(12, 3, n1).clip(5, 25)
fc1 = np.random.normal(10, 4, n1).clip(2, 25)

# Zone 2: Soft clay (SW quadrant) — elongated cluster
n2 = 30
x2 = np.random.normal(25, 15, n2)
y2 = np.random.normal(30, 7, n2)
N2 = np.random.normal(8, 3, n2).clip(2, 15)
wc2 = np.random.normal(45, 8, n2).clip(25, 70)
fc2 = np.random.normal(65, 10, n2).clip(40, 90)

# Zone 3: Silty sand (center) — loose cluster
n3 = 25
x3 = np.random.normal(55, 12, n3)
y3 = np.random.normal(55, 12, n3)
N3 = np.random.normal(20, 6, n3).clip(10, 35)
wc3 = np.random.normal(22, 5, n3).clip(10, 35)
fc3 = np.random.normal(35, 8, n3).clip(15, 55)

# Outliers: anomalous boreholes
n_out = 15
x_out = np.random.uniform(0, 100, n_out)
y_out = np.random.uniform(0, 100, n_out)
N_out = np.random.uniform(2, 50, n_out)
wc_out = np.random.uniform(5, 70, n_out)
fc_out = np.random.uniform(2, 90, n_out)

# Combine all zones
df = pd.DataFrame({
    'x': np.round(np.concatenate([x1, x2, x3, x_out]), 1),
    'y': np.round(np.concatenate([y1, y2, y3, y_out]), 1),
    'N_value': np.round(np.concatenate([N1, N2, N3, N_out]), 0).astype(int),
    'water_content': np.round(np.concatenate([wc1, wc2, wc3, wc_out]), 1),
    'fines_content': np.round(np.concatenate([fc1, fc2, fc3, fc_out]), 1)
})

# True labels for evaluation (not used in clustering)
true_zones = np.concatenate([np.full(n1, 0), np.full(n2, 1), np.full(n3, 2), np.full(n_out, -1)])

print(f"Dataset shape: {df.shape}")
print(f"\nSummary statistics:")
df.describe().round(1)

In [ ]:
# ─────────────────────────────────────────────────────
# VISUALIZE RAW DATA
# ─────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, feature, cmap in zip(axes,
    ['N_value', 'water_content', 'fines_content'],
    ['RdYlGn', 'Blues', 'Oranges']):
    sc = ax.scatter(df['x'], df['y'], c=df[feature], cmap=cmap, s=40, alpha=0.7, edgecolors='gray', linewidth=0.5)
    plt.colorbar(sc, ax=ax, label=feature)
    ax.set_xlabel('X (m)')
    ax.set_ylabel('Y (m)')
    ax.set_title(f'Borehole {feature}')
    ax.grid(True, alpha=0.3)

fig.suptitle('Site Investigation: Borehole Data', fontsize=14)
plt.tight_layout()
plt.show()

print("Visual inspection suggests 2-3 distinct zones plus scattered outliers.")
print("NE: high N-value, low water content (dense sand)")
print("SW: low N-value, high water content (soft clay)")

---

## Part 2: DBSCAN Clustering

DBSCAN (Density-Based Spatial Clustering of Applications with Noise) has two parameters:

- **eps**: Maximum distance between two points in the same neighborhood
- **min_samples**: Minimum points required to form a dense region (core point)

DBSCAN classifies each point as:
- **Core point**: Has at least `min_samples` neighbors within `eps`
- **Border point**: Within `eps` of a core point but not a core point itself
- **Noise point**: Neither core nor border (labeled as -1)

Advantages over K-Means:
- No need to specify number of clusters
- Can find arbitrarily shaped clusters
- Automatically identifies outliers

In [ ]:
# ─────────────────────────────────────────────────────
# SCALE FEATURES AND RUN DBSCAN
# ─────────────────────────────────────────────────────

features = ['x', 'y', 'N_value', 'water_content', 'fines_content']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[features])

# DBSCAN with tuned parameters
db = DBSCAN(eps=0.9, min_samples=5)
db_labels = db.fit_predict(X_scaled)

n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise = (db_labels == -1).sum()

print(f"DBSCAN Results:")
print(f"  Clusters found: {n_clusters}")
print(f"  Noise points:   {n_noise}")
print(f"  Core samples:   {len(db.core_sample_indices_)}")
print(f"\nCluster sizes:")
for label in sorted(set(db_labels)):
    count = (db_labels == label).sum()
    name = 'Noise' if label == -1 else f'Cluster {label}'
    print(f"  {name}: {count} boreholes")

In [ ]:
# ─────────────────────────────────────────────────────
# VISUALIZE DBSCAN RESULTS
# ─────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# DBSCAN clusters on spatial map
unique_labels = sorted(set(db_labels))
colors_db = plt.cm.tab10(np.linspace(0, 1, len(unique_labels)))

for label, color in zip(unique_labels, colors_db):
    mask = db_labels == label
    marker = 'x' if label == -1 else 'o'
    name = 'Noise/Outlier' if label == -1 else f'Zone {label}'
    alpha = 0.4 if label == -1 else 0.7
    axes[0].scatter(df['x'][mask], df['y'][mask], c=[color], marker=marker,
                    s=50, label=name, alpha=alpha, edgecolors='gray', linewidth=0.5)

axes[0].set_xlabel('X (m)')
axes[0].set_ylabel('Y (m)')
axes[0].set_title('DBSCAN: Soil Zones & Outliers')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Compare with true zones
true_colors = {-1: 'gray', 0: 'tab:blue', 1: 'tab:orange', 2: 'tab:green'}
for label in sorted(set(true_zones)):
    mask = true_zones == label
    name = 'Scattered' if label == -1 else f'True Zone {label}'
    axes[1].scatter(df['x'][mask], df['y'][mask], c=true_colors[label],
                    s=50, label=name, alpha=0.7, edgecolors='gray', linewidth=0.5)

axes[1].set_xlabel('X (m)')
axes[1].set_ylabel('Y (m)')
axes[1].set_title('True Soil Zones (for reference)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("DBSCAN successfully identified the main soil zones and flagged")
print("scattered boreholes as noise/outliers \u2014 without specifying k!")

---

## Part 3: Effect of eps and min_samples

In [ ]:
# ─────────────────────────────────────────────────────
# SENSITIVITY TO eps AND min_samples
# ─────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

eps_values = [0.5, 0.9, 1.5]
min_samples_values = [3, 5]

for row, ms in enumerate(min_samples_values):
    for col, eps in enumerate(eps_values):
        db_test = DBSCAN(eps=eps, min_samples=ms)
        labels = db_test.fit_predict(X_scaled)
        n_clust = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise_t = (labels == -1).sum()

        ax = axes[row, col]
        scatter = ax.scatter(df['x'], df['y'], c=labels, cmap='tab10',
                            s=30, alpha=0.7, edgecolors='gray', linewidth=0.3)
        ax.set_title(f'eps={eps}, min_samples={ms}\n{n_clust} clusters, {n_noise_t} noise')
        ax.set_xlabel('X (m)')
        ax.set_ylabel('Y (m)')
        ax.grid(True, alpha=0.3)

fig.suptitle('DBSCAN Sensitivity Analysis', fontsize=14)
plt.tight_layout()
plt.show()

print("Small eps: many small clusters (or all noise)")
print("Large eps: fewer, larger clusters (may merge distinct zones)")
print("Higher min_samples: stricter density requirement (more noise points)")

---

## Part 4: Hierarchical Clustering & Dendrogram

In [ ]:
# ─────────────────────────────────────────────────────
# HIERARCHICAL CLUSTERING WITH DENDROGRAM
# ─────────────────────────────────────────────────────

# Compute linkage matrix
Z = linkage(X_scaled, method='ward')

fig, ax = plt.subplots(figsize=(14, 6))
dendrogram(
    Z,
    truncate_mode='lastp',
    p=20,
    leaf_rotation=90,
    leaf_font_size=9,
    ax=ax
)
ax.set_xlabel('Cluster (or sample count)')
ax.set_ylabel('Distance (Ward)')
ax.set_title('Dendrogram: Hierarchical Clustering of Borehole Data')
ax.axhline(y=10, color='r', linestyle='--', alpha=0.7, label='Cut at 3 clusters')
ax.legend()
plt.tight_layout()
plt.show()

print("The dendrogram shows how boreholes merge into clusters.")
print("Long vertical lines = large distance between merges = natural cluster boundary.")
print("Cut the dendrogram horizontally to choose the number of clusters.")

In [ ]:
# ─────────────────────────────────────────────────────
# AGGLOMERATIVE CLUSTERING WITH 3 CLUSTERS
# ─────────────────────────────────────────────────────

hc = AgglomerativeClustering(n_clusters=3, linkage='ward')
hc_labels = hc.fit_predict(X_scaled)

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(df['x'], df['y'], c=hc_labels, cmap='tab10',
                     s=50, alpha=0.7, edgecolors='gray', linewidth=0.5)
ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_title('Hierarchical Clustering (Ward, k=3)')
plt.colorbar(scatter, ax=ax, label='Cluster')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Cluster statistics
df_hc = df.copy()
df_hc['cluster'] = hc_labels
print("Hierarchical Clustering: Zone Properties")
print("=" * 60)
for c in sorted(set(hc_labels)):
    subset = df_hc[df_hc['cluster'] == c]
    print(f"\nZone {c} ({len(subset)} boreholes):")
    print(f"  N-value:       {subset['N_value'].mean():.1f} (std={subset['N_value'].std():.1f})")
    print(f"  Water content: {subset['water_content'].mean():.1f}% (std={subset['water_content'].std():.1f})")
    print(f"  Fines content: {subset['fines_content'].mean():.1f}% (std={subset['fines_content'].std():.1f})")

---

## Part 5: Compare K-Means vs DBSCAN vs Hierarchical

In [ ]:
# ─────────────────────────────────────────────────────
# THREE-WAY COMPARISON
# ─────────────────────────────────────────────────────

# K-Means
km = KMeans(n_clusters=3, random_state=42, n_init=10)
km_labels = km.fit_predict(X_scaled)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

methods = [
    ('K-Means (k=3)', km_labels),
    ('DBSCAN (eps=0.9)', db_labels),
    ('Hierarchical (k=3)', hc_labels)
]

for ax, (name, labels) in zip(axes, methods):
    scatter = ax.scatter(df['x'], df['y'], c=labels, cmap='tab10',
                         s=50, alpha=0.7, edgecolors='gray', linewidth=0.5)
    ax.set_xlabel('X (m)')
    ax.set_ylabel('Y (m)')
    n_c = len(set(labels)) - (1 if -1 in labels else 0)
    n_n = (np.array(labels) == -1).sum()
    ax.set_title(f'{name}\n{n_c} clusters, {n_n} noise')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Silhouette scores (excluding noise for DBSCAN)
print("Silhouette Score Comparison (higher is better):")
print("=" * 50)
print(f"  K-Means:      {silhouette_score(X_scaled, km_labels):.4f}")

non_noise = db_labels != -1
if non_noise.sum() > 1 and len(set(db_labels[non_noise])) > 1:
    print(f"  DBSCAN:       {silhouette_score(X_scaled[non_noise], db_labels[non_noise]):.4f} (excluding noise)")

print(f"  Hierarchical: {silhouette_score(X_scaled, hc_labels):.4f}")

In [ ]:
# ─────────────────────────────────────────────────────
# COMPARISON TABLE
# ─────────────────────────────────────────────────────

print("Algorithm Comparison for Soil Zonation")
print("=" * 70)
print(f"{'Property':<30s} {'K-Means':>12s} {'DBSCAN':>12s} {'Hierarchical':>12s}")
print("-" * 70)
print(f"{'Specify k?':<30s} {'Yes':>12s} {'No':>12s} {'Yes':>12s}")
print(f"{'Handles outliers?':<30s} {'No':>12s} {'Yes':>12s} {'No':>12s}")
print(f"{'Cluster shapes':<30s} {'Spherical':>12s} {'Arbitrary':>12s} {'Any':>12s}")
print(f"{'Dendrogram?':<30s} {'No':>12s} {'No':>12s} {'Yes':>12s}")
print(f"{'Scalability':<30s} {'Good':>12s} {'Medium':>12s} {'Poor':>12s}")
print(f"{'Deterministic?':<30s} {'No':>12s} {'Yes':>12s} {'Yes':>12s}")
print()
print("For geotechnical zonation:")
print("  - DBSCAN is best when you expect outlier boreholes")
print("  - Hierarchical is best for understanding zone relationships")
print("  - K-Means is simplest when you know how many zones to expect")

---

## Part 6: Outlier Analysis with DBSCAN

In [ ]:
# ─────────────────────────────────────────────────────
# EXAMINE OUTLIER BOREHOLES
# ─────────────────────────────────────────────────────

outlier_mask = db_labels == -1
df_outliers = df[outlier_mask].copy()
df_normal = df[~outlier_mask].copy()

print(f"DBSCAN identified {outlier_mask.sum()} outlier boreholes")
print(f"\nOutlier borehole properties:")
print("=" * 50)
print(df_outliers.describe().round(1))

print(f"\nNormal borehole properties:")
print("=" * 50)
print(df_normal.describe().round(1))

print(f"\nOutliers tend to have more variable properties and are")
print(f"spatially isolated \u2014 they may need additional investigation.")

---

## Practice Exercises

### Exercise 1: Optimal eps
Plot the k-distance graph (sort the distance to the 5th nearest neighbor for each point). The "elbow" suggests a good eps value. Does it match our choice of 0.9?

### Exercise 2: Different Linkage Methods
Try hierarchical clustering with `linkage='complete'` and `linkage='average'`. Compare the dendrograms. Which gives the clearest cluster separation?

### Exercise 3: Feature Subsets
Run DBSCAN using only spatial features (x, y) vs only soil properties (N_value, water_content, fines_content). How do the clusters differ?

In [ ]:
# EXERCISE 1: k-distance graph
# from sklearn.neighbors import NearestNeighbors
# nn = NearestNeighbors(n_neighbors=5)
# nn.fit(X_scaled)
# distances, _ = nn.kneighbors(X_scaled)
# k_dist = np.sort(distances[:, -1])[::-1]
# plt.plot(k_dist)
# plt.xlabel('Points (sorted)')
# plt.ylabel('5th nearest neighbor distance')
# plt.title('k-Distance Graph for eps Selection')
# plt.show()

In [ ]:
# EXERCISE 2: Different linkage methods
# for method in ['ward', 'complete', 'average']:
#     Z_m = linkage(X_scaled, method=method)
#     fig, ax = plt.subplots(figsize=(12, 4))
#     dendrogram(Z_m, truncate_mode='lastp', p=15, ax=ax)
#     ax.set_title(f'Dendrogram: {method} linkage')
#     plt.tight_layout()
#     plt.show()

---

## Key Takeaways

1. **DBSCAN finds clusters without specifying k** and automatically detects outliers as noise points. It works well for soil data where anomalous boreholes are common.

2. **eps and min_samples control DBSCAN behavior**. Too small eps fragments zones; too large eps merges distinct zones. The k-distance graph helps choose eps.

3. **Hierarchical clustering reveals nested structure** through dendrograms. Engineers can choose the number of zones by cutting the dendrogram at different heights.

4. **No single algorithm is best for all data**. K-Means works for well-separated spherical clusters, DBSCAN handles noise and irregular shapes, and hierarchical clustering shows relationships at multiple scales.

5. **Outlier detection is a bonus feature** of DBSCAN. Isolated boreholes flagged as noise may indicate localized geological anomalies requiring additional investigation.

---

## How This Connects to ML

| What We Did Today | ML Equivalent |
|-------------------|---------------|
| Soil zone identification | Unsupervised clustering |
| DBSCAN on spatial soil data | Density-based clustering |
| Outlier borehole detection | Anomaly detection |
| Dendrogram interpretation | Hierarchical agglomerative clustering |
| K-Means vs DBSCAN vs Hierarchical | Algorithm selection for clustering |
| eps sensitivity analysis | Hyperparameter tuning |
| Silhouette score comparison | Cluster quality evaluation |

---

*Day 047 of 100 — K-Means forces every point into a cluster; DBSCAN knows when to say "that one's just weird."*